# GI Cancer Clinical Trials — Document-Complete Analysis (No SEER)

This notebook reproduces all analyses described in your document **except SEER incidence**. It generates a *complete* page-2 table with additional columns (distance metrics and multi-radius coverage), plus supporting summaries and QC outputs.


In [68]:
# %pip install pandas geopandas shapely scikit-learn requests tqdm xlsxwriter

import os, re, glob, time, requests
import numpy as np
import pandas as pd
import geopandas as gpd
from sklearn.neighbors import BallTree
from shapely.geometry import Point
from tqdm import tqdm
tqdm.pandas()

DATA_DIR = "../data"
TRAIL_DIR = "../data/output/test"
OUT_DIR = "out"
ZCTA_DIR = os.path.join(DATA_DIR, "tl_2020_us_zcta510")
RUCA_CSV = os.path.join(DATA_DIR, "RUCA-codes-2020-zipcode.csv")
TRIAL_FILES = sorted(glob.glob(os.path.join(TRAIL_DIR, "*_with_zip.csv")))

for f in TRIAL_FILES:
    print(f)

os.makedirs(OUT_DIR, exist_ok=True)

USE_DECENNIAL_2020 = False
CENSUS_API_KEY = os.environ.get("CENSUS_API_KEY", "")


../data/output/test/Bispecific_with_zip.csv
../data/output/test/CAR_with_zip.csv


In [69]:
import re
ZIP_RE_5 = re.compile(r"^\d{5}$")
ZIP_RE_FINDALL = re.compile(r"\b\d{5}\b")

def to_zip5(x):
    if pd.isna(x): return None
    s = re.sub(r"[^\d]", "", str(x).strip())
    if len(s) >= 5: s = s[:5]
    return s if ZIP_RE_5.match(s) else None

def extract_zips_from_locations(text):
    if pd.isna(text): return []
    return list(dict.fromkeys(ZIP_RE_FINDALL.findall(str(text))))

def classify_rural_urban(primary_ruca):
    try: p = float(primary_ruca)
    except: return None
    return "Urban" if 1 <= p < 4 else "Rural"

def pct1(x): return f"{100*x:.1f}%" if pd.notna(x) else "NA"


In [70]:
print("Loading RUCA (robust)…")
ruca_raw = pd.read_csv(RUCA_CSV, dtype=str)

def norm(c): return re.sub(r"[^a-z0-9]+", "", str(c).strip().lower())
cols_norm = [norm(c) for c in ruca_raw.columns]
from collections import defaultdict
counts = defaultdict(int); newcols=[]
for c in cols_norm:
    counts[c]+=1; newcols.append(c if counts[c]==1 else f"{c}_{counts[c]}")
ruca_raw.columns = newcols

# ZIP column by content
zip_col = None
for c in ruca_raw.columns:
    m = ruca_raw[c].astype(str).str.extract(r"(\d{5})")[0]
    if m.notna().mean() >= 0.9: zip_col = c; break
if not zip_col:
    for hint in ["zipcode","zip","zip_code","zip5","zcta5","zcta5ce10"]:
        if hint in ruca_raw.columns: zip_col = hint; break
if not zip_col: raise ValueError("Could not identify ZIP column in RUCA.")

# RUCA code column by range 1..10
ruca_code_col = None
for c in ruca_raw.columns:
    s = pd.to_numeric(ruca_raw[c], errors="coerce")
    if s.notna().any() and (s.dropna().between(1,10).mean() >= 0.8):
        ruca_code_col = c; break
if not ruca_code_col:
    for hint in ["primaryruca","ruca","rucacode","primaryrucacode"]:
        if hint in ruca_raw.columns: ruca_code_col = hint; break
if not ruca_code_col: raise ValueError("Could not identify Primary RUCA code column in RUCA.")

state_col = None
for hint in ["state","stateabbr","statecode","stusab"]:
    if hint in ruca_raw.columns: state_col = hint; break

ruca = pd.DataFrame({
    "zip": ruca_raw[zip_col].astype(str).str.extract(r"(\d{5})")[0].str.zfill(5),
    "PrimaryRUCA": ruca_raw[ruca_code_col].astype(str)
})
ruca["PrimaryRUCA_num"] = pd.to_numeric(ruca["PrimaryRUCA"], errors="coerce")
ruca["rural_urban"] = ruca["PrimaryRUCA_num"].apply(classify_rural_urban)
ruca["State"] = ruca_raw[state_col] if state_col else np.nan
ruca = ruca.dropna(subset=["zip"]).drop_duplicates(subset=["zip"])
print("RUCA loaded:", len(ruca))


Loading RUCA (robust)…
RUCA loaded: 41146


In [71]:
print("Loading ZCTA shapefile & centroids…")
zcta = gpd.read_file(ZCTA_DIR).to_crs(epsg=4326)
id_col = "ZCTA5CE10" if "ZCTA5CE10" in zcta.columns else None
assert id_col is not None, "ZCTA id column not found"
zcta["centroid"] = zcta.geometry.centroid
zcta["lon"] = zcta["centroid"].x; zcta["lat"] = zcta["centroid"].y
zip_centroids = zcta[[id_col,"lat","lon"]].rename(columns={id_col:"zip"})
zip_centroids["zip"] = zip_centroids["zip"].astype(str).str.zfill(5)
print("ZCTAs:", len(zip_centroids))


Loading ZCTA shapefile & centroids…


/var/folders/4l/64tth9h948n3dbq1py15y2qr0000gn/T/ipykernel_3388/3421087281.py:5: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  zcta["centroid"] = zcta.geometry.centroid


ZCTAs: 33144


In [72]:
print("Loading GI trials ZIPs…")
def load_trials_one(path):
    df = pd.read_csv(path, dtype=str)
    zip_col = next((c for c in ["zip","zipcode","Zipcode","ZIP","Zip","ZIPCode","postal_code"] if c in df.columns), None)
    if zip_col:
        df["zip"] = df[zip_col].apply(to_zip5); df = df.dropna(subset=["zip"])
    elif "Locations" in df.columns:
        df["__zips"] = df["Locations"].apply(extract_zips_from_locations)
        df = df.explode("__zips"); df["zip"] = df["__zips"].apply(to_zip5)
        df = df.dropna(subset=["zip"]).drop(columns="__zips")
    else:
        raise ValueError(f"No ZIP or Locations in {os.path.basename(path)}")
    stem = re.sub(r"_with_zip.*$", "", os.path.basename(path).lower()).replace(".csv","")
    df["cancer_type"] = stem
    for c in ["NCT Number","Study Title","Conditions","Interventions","Sponsor","Locations"]:
        if c not in df.columns: df[c]=np.nan
    return df[["NCT Number","Study Title","Conditions","Interventions","Sponsor","Locations","zip","cancer_type"]]

parts=[] 
for f in TRIAL_FILES:
    try:
        p = load_trials_one(f); parts.append(p)
        print(f" - {os.path.basename(f)} → {len(p):,} rows (post-zip)")
    except Exception as e:
        print("Skip", os.path.basename(f), ":", e)

trials = pd.concat(parts, ignore_index=True) if parts else pd.DataFrame(columns=["zip","cancer_type"])
trials = trials.drop_duplicates(subset=["zip","cancer_type"]).reset_index(drop=True)
print("Unique trial ZIP entries:", len(trials))


Loading GI trials ZIPs…
 - Bispecific_with_zip.csv → 681 rows (post-zip)
 - CAR_with_zip.csv → 1,051 rows (post-zip)
Unique trial ZIP entries: 417


In [73]:
print("Geocoding trial sites…")
trial_sites = trials.merge(zip_centroids, on="zip", how="left")
miss = trial_sites["lat"].isna().sum()
print("Missing lat/lon:", miss)
trial_sites = trial_sites.dropna(subset=["lat","lon"]).drop_duplicates(subset=["zip","cancer_type"])
print("Geocoded trial sites:", len(trial_sites))


Geocoding trial sites…
Missing lat/lon: 0
Geocoded trial sites: 417


In [74]:
print("Fetching population & equity variables…")
import time

ACS_DIR = os.path.join(DATA_DIR, "acs")
os.makedirs(ACS_DIR, exist_ok=True)

def fetch_acs_b01003(year):
    base = f"https://api.census.gov/data/{year}/acs/acs5"
    params = {"get": "NAME,B01003_001E", "for": "zip code tabulation area:*"}
    if CENSUS_API_KEY:
        params["key"] = CENSUS_API_KEY
    r = requests.get(base, params=params, timeout=60)
    r.raise_for_status()
    rows = r.json()
    cols = rows[0]; data = rows[1:]
    df = (
        pd.DataFrame(data, columns=cols)
          .rename(columns={"zip code tabulation area": "zip", "B01003_001E": "population"})
    )
    df["zip"] = df["zip"].astype(str).str.zfill(5)
    df["population"] = pd.to_numeric(df["population"], errors="coerce")
    df["year"] = year
    return df[["zip", "population", "year"]]

# ---- Build population (Decennial 2020 optional; otherwise ACS 2019–2023) ----
pop_df = None

if USE_DECENNIAL_2020:
    try:
        url = "https://www2.census.gov/geo/docs/maps-data/data/gazetteer/2020_Gazetteer/2020_Gaz_zcta5_national.txt"
        p = os.path.join(ACS_DIR, "gazetteer_2020_zcta.txt")
        r = requests.get(url, timeout=120); r.raise_for_status()
        open(p, "wb").write(r.content)
        g = pd.read_csv(p, sep="|", dtype=str)
        g["zip"] = g["ZCTA5"].astype(str).str.zfill(5)
        pop_col = None
        for c in ["POP20", "POPPT", "POPULATION", "POP"]:
            if c in g.columns:
                pop_col = c; break
        if pop_col is not None:
            g["population"] = pd.to_numeric(g[pop_col], errors="coerce")
            pop_df = g[["zip", "population"]].copy()
            print("Using Decennial 2020 Gazetteer population.")
        else:
            print("Gazetteer has no recognized population column; falling back to ACS.")
    except Exception as e:
        print("Decennial 2020 download/parse failed; falling back to ACS:", e)

if pop_df is None:
    years = [2023, 2022, 2021, 2020, 2019]
    parts = []
    for y in years:
        try:
            parts.append(fetch_acs_b01003(y))
            time.sleep(0.25)
        except Exception as e:
            print("ACS fetch failed for", y, ":", e)

    if not parts:
        raise RuntimeError("ACS population could not be fetched for any year.")

    allp = pd.concat(parts, ignore_index=True)
    # keep the latest available year per ZIP
    allp = allp.sort_values(["zip", "year"], ascending=[True, False])
    pop_df = allp.drop_duplicates("zip", keep="first")[["zip", "population"]]

print("Population rows:", len(pop_df))

# ---- Equity variables (ACS 2023) ----
def fetch_acs_vars(year, vars_list):
    base = f"https://api.census.gov/data/{year}/acs/acs5"
    params = {"get": ",".join(["NAME"] + vars_list), "for": "zip code tabulation area:*"}
    if CENSUS_API_KEY:
        params["key"] = CENSUS_API_KEY
    r = requests.get(base, params=params, timeout=60)
    r.raise_for_status()
    rows = r.json(); cols = rows[0]; data = rows[1:]
    df = pd.DataFrame(data, columns=cols).rename(columns={"zip code tabulation area": "zip"})
    df["zip"] = df["zip"].astype(str).str.zfill(5)
    for v in vars_list:
        df[v] = pd.to_numeric(df[v], errors="coerce")
    return df[["zip"] + vars_list]

YEAR = 2023
inc = fetch_acs_vars(YEAR, ["B19013_001E"]).rename(columns={"B19013_001E": "mhi"})
eth = fetch_acs_vars(YEAR, ["B03003_001E","B03003_003E"]).rename(
    columns={"B03003_001E": "pop_total_eth", "B03003_003E": "pop_hispanic"}
)
blk = fetch_acs_vars(YEAR, ["B02001_001E","B02001_003E"]).rename(
    columns={"B02001_001E": "pop_total_race", "B02001_003E": "pop_black"}
)


Fetching population & equity variables…
Population rows: 33971


In [75]:
print("Building ZIP universe…")
zip_universe = zip_centroids.merge(ruca[["zip","PrimaryRUCA","rural_urban","State"]], on="zip", how="left")
zip_universe = zip_universe.merge(pop_df, on="zip", how="left")
zip_universe = zip_universe.merge(inc, on="zip", how="left").merge(eth, on="zip", how="left").merge(blk, on="zip", how="left")
print("Rows:", len(zip_universe), "| pop present:", f"{100*zip_universe['population'].notna().mean():.1f}%")


Building ZIP universe…
Rows: 33144 | pop present: 99.9%


In [76]:
print("Computing nearest distances and coverage flags…")
EARTH_RADIUS_MILES = 3958.7613

def build_tree(df):
    pts = np.deg2rad(df[["lat","lon"]].to_numpy())
    return BallTree(pts, metric="haversine"), pts

if len(trial_sites)==0: raise ValueError("No trial sites after geocoding.")

any_tree, any_pts = build_tree(trial_sites)
trees_by_cancer = {c: build_tree(g) for c, g in trial_sites.groupby("cancer_type")}

qpts = np.deg2rad(zip_universe[["lat","lon"]].to_numpy())

# nearest distance to any GI
dist_rad, ind = any_tree.query(qpts, k=1)
zip_universe["dist_any_gi_mi"] = (dist_rad[:,0] * EARTH_RADIUS_MILES)

# nearest distance per cancer;  multi-radius flags per cancer and overall
RADII = [30, 50, 60, 120]
def within_radius(tree, r_mi):
    r_rad = r_mi / EARTH_RADIUS_MILES
    inds = tree.query_radius(qpts, r=r_rad)
    return np.array([1 if len(ix)>0 else 0 for ix in inds], dtype=int)

# Any GI flags
for r in RADII:
    zip_universe[f"covered_any_gi_{r}mi"] = within_radius(any_tree, r)

# Cancer-specific distances & flags
for cancer, (tree, pts) in trees_by_cancer.items():
    # distance
    d_rad, _ = tree.query(qpts, k=1)
    col_d = re.sub(r"[^a-z0-9_]+","_", f"dist_{cancer}_mi".lower())
    zip_universe[col_d] = d_rad[:,0] * EARTH_RADIUS_MILES
    # flags at radii
    for r in RADII:
        col = re.sub(r"[^a-z0-9_]+","_", f"covered_{cancer}_{r}mi".lower())
        zip_universe[col] = within_radius(tree, r)


Computing nearest distances and coverage flags…


In [77]:
print("Deriving income quartiles (population-weighted)…")
def weighted_quantile(values, weights, qs):
    v = np.asarray(values); w = np.asarray(weights)
    idx = np.argsort(v); v=v[idx]; w=w[idx]
    cw = np.cumsum(w)-0.5*w; cw = cw/np.sum(w)
    return np.interp(qs, cw, v)

m = zip_universe["mhi"].notna() & zip_universe["population"].notna()
if m.any():
    q1,q3 = weighted_quantile(zip_universe.loc[m,"mhi"], zip_universe.loc[m,"population"], [0.25,0.75])
else:
    q1=q3=np.nan
zip_universe["income_q"]=pd.NA
zip_universe.loc[zip_universe["mhi"].notna() & (zip_universe["mhi"]<=q1), "income_q"] = "Q1 (Lowest)"
zip_universe.loc[zip_universe["mhi"].notna() & (zip_universe["mhi"]>=q3), "income_q"] = "Q4 (Highest)"
print("Q1:", q1, "Q4:", q3)


Deriving income quartiles (population-weighted)…
Q1: 60895.484005838174 Q4: 101724.01291532787


In [78]:
print("Building COMPLETE Page-2 table (no SEER)…")

cov_to_label = {
    "covered_any_gi": "Any GI",
    "covered_bispecific":"Bispecific",
    "covered_CAR":"CAR",

}
label_to_key = {v:k.replace("covered_","") for k,v in cov_to_label.items() if k!="covered_any_gi"}

def pop_share(df, flag, w="population"):
    if w in df.columns and df[w].notna().any():
        num = (df[flag]*df[w]).sum(skipna=True)
        den = df[w].sum(skipna=True)
        return num, den, (num/den if den>0 else np.nan)
    return np.nan, np.nan, df[flag].mean()

# Trials & sites counts
def load_trials_raw(path):
    print("--------")
    print(path)
    df = pd.read_csv(path, dtype=str)
    stem = re.sub(r"_with_zip.*$","", os.path.basename(path).lower()).replace(".csv","")
    df["cancer_type"]=stem
    return df

raw_list=[]
for f in TRIAL_FILES:
    try: raw_list.append(load_trials_raw(f))
    except Exception as e: print("Skip", os.path.basename(f), e)
trials_raw = pd.concat(raw_list, ignore_index=True) if raw_list else pd.DataFrame(columns=["NCT Number","cancer_type"])
nct_per = trials_raw.groupby("cancer_type")["NCT Number"].nunique() if "NCT Number" in trials_raw.columns else pd.Series(dtype=int)
nct_any = trials_raw["NCT Number"].nunique() if "NCT Number" in trials_raw.columns else np.nan

sites_per = trial_sites.groupby("cancer_type")["zip"].nunique() if len(trial_sites) else pd.Series(dtype=int)
sites_any = trial_sites["zip"].nunique() if len(trial_sites) else np.nan

# Assemble
cancers = ["any_gi","bispecific","CAR"]

rows=[]
for key in cancers:
    label = "Any GI" if key=="any_gi" else cov_to_label[f"covered_{key}"]
    # distance column name
    dist_col = f"dist_{key}_mi" if key!="any_gi" else "dist_any_gi_mi"
    # coverage flags at radii
    col30 = f"covered_{key}_30mi"
    col50 = f"covered_{key}_50mi"
    col60 = f"covered_{key}_60mi"
    col120= f"covered_{key}_120mi"
    # Some flags may not exist (for cancer types) if no sites; guard:
    for c in [col30,col50,col60,col120]:
        if c not in zip_universe.columns: zip_universe[c]=np.nan

    # overall shares & covered population (50mi)
    num50, den50, share50 = pop_share(zip_universe, col50, "population")
    coveredM = (num50/1e6) if pd.notna(num50) else np.nan

    # multi-radius %
    _,_, share30 = pop_share(zip_universe, col30, "population")
    _,_, share60 = pop_share(zip_universe, col60, "population")
    _,_, share120= pop_share(zip_universe, col120, "population")

    # urban/rural at 50mi
    urb = zip_universe["rural_urban"].eq("Urban"); rur = zip_universe["rural_urban"].eq("Rural")
    _,_, share50_urb = pop_share(zip_universe[urb], col50, "population")
    _,_, share50_rur = pop_share(zip_universe[rur], col50, "population")

    # income quartiles at 50mi
    q1 = zip_universe["income_q"].eq("Q1 (Lowest)"); q4 = zip_universe["income_q"].eq("Q4 (Highest)")
    _,_, share50_q1 = pop_share(zip_universe[q1], col50, "population")
    _,_, share50_q4 = pop_share(zip_universe[q4], col50, "population")

    # race ethnicity at 50mi (Hispanic / Black)
    def subgroup_share(flag, sub_pop):
        d=zip_universe.copy()
        if sub_pop not in d.columns: return np.nan
        num=(d[flag]*d[sub_pop]).sum(skipna=True); den=d[sub_pop].sum(skipna=True)
        return num/den if den>0 else np.nan
    share50_hisp = subgroup_share(col50, "pop_hispanic")
    share50_black= subgroup_share(col50, "pop_black")

    # population-weighted median distance (nearest site)
    if dist_col in zip_universe.columns and zip_universe["population"].notna().any():
        d = zip_universe[[dist_col,"population"]].dropna()
        if len(d):
            order = np.argsort(d[dist_col].values)
            v = d[dist_col].values[order]; w = d["population"].values[order]
            cw = np.cumsum(w)/w.sum()
            idx = np.searchsorted(cw, 0.5, side="left")
            med_dist = float(v[min(idx, len(v)-1)])
        else:
            med_dist = np.nan
    else:
        med_dist = np.nan

    # trials & sites
    if key=="any_gi":
        n_trials = nct_any; n_sites = sites_any
    else:
        n_trials = nct_per.get(key, np.nan); n_sites = sites_per.get(key, np.nan)

    rows.append({
        "Cancer Type": label,
        "No. Trials (NCT IDs)": int(n_trials) if pd.notna(n_trials) else np.nan,
        "No. Sites (ZIPs)": int(n_sites) if pd.notna(n_sites) else np.nan,
        "Median Distance to Nearest Site (mi)": med_dist,
        "% Pop ≤30 mi": share30, "% Pop ≤50 mi": share50, "% Pop ≤60 mi": share60, "% Pop ≤120 mi": share120,
        "Covered Population at 50 mi (M)": coveredM,
        "% Urban Pop ≤50 mi": share50_urb, "% Rural Pop ≤50 mi": share50_rur,
        "% Lowest Income Quartile ≤50 mi": share50_q1, "% Highest Income Quartile ≤50 mi": share50_q4,
        "% Hispanic Pop ≤50 mi": share50_hisp, "% Black Pop ≤50 mi": share50_black
    })

table = pd.DataFrame(rows)

# Order rows: Any GI first
mask_any = table["Cancer Type"].eq("Any GI")
table = pd.concat([table[mask_any], table[~mask_any].sort_values("Cancer Type")], ignore_index=True)

# Format copy
fmt = table.copy()
for c in ["% Pop ≤30 mi","% Pop ≤50 mi","% Pop ≤60 mi","% Pop ≤120 mi",
          "% Urban Pop ≤50 mi","% Rural Pop ≤50 mi",
          "% Lowest Income Quartile ≤50 mi","% Highest Income Quartile ≤50 mi",
          "% Hispanic Pop ≤50 mi","% Black Pop ≤50 mi"]:
    fmt[c] = fmt[c].map(lambda x: f"{100*x:.1f}%" if pd.notna(x) else "NA")
fmt["Covered Population at 50 mi (M)"] = fmt["Covered Population at 50 mi (M)"].map(lambda x: f"{x:.2f}" if pd.notna(x) else "NA")
fmt["Median Distance to Nearest Site (mi)"] = fmt["Median Distance to Nearest Site (mi)"].map(lambda x: f"{x:.1f}" if pd.notna(x) else "NA")

# Save
raw_path = os.path.join(OUT_DIR, "test_page2_table_complete_raw.csv")
fmt_path = os.path.join(OUT_DIR, "test_page2_table_complete_formatted.csv")
xlsx_path= os.path.join(OUT_DIR, "test_page2_table_complete.xlsx")
table.to_csv(raw_path, index=False)
fmt.to_csv(fmt_path, index=False)
with pd.ExcelWriter(xlsx_path, engine="xlsxwriter") as w:
    fmt.to_excel(w, sheet_name="test_Page2_Table", index=False)

print("Saved:")
print(" -", raw_path)
print(" -", fmt_path)
print(" -", xlsx_path)


Building COMPLETE Page-2 table (no SEER)…
--------
../data/output/test/Bispecific_with_zip.csv
--------
../data/output/test/CAR_with_zip.csv
Saved:
 - out/test_page2_table_complete_raw.csv
 - out/test_page2_table_complete_formatted.csv
 - out/test_page2_table_complete.xlsx


In [12]:
print("Building supporting summaries…")
# State-level Any GI 50mi
state = zip_universe.dropna(subset=["State"]).copy()
def pop_share(df, flag, w="population"):
    if w in df.columns and df[w].notna().any():
        num=(df[flag]*df[w]).sum(skipna=True); den=df[w].sum(skipna=True)
        return num/den if den>0 else np.nan
    return df[flag].mean()

state_any50 = state.groupby(["State","rural_urban"]).apply(lambda g: pd.Series({
    "n_zips": len(g),
    "% Pop ≤50 mi": pop_share(g, "covered_any_gi_50mi", "population")
})).reset_index()
state_any50.to_csv(os.path.join(OUT_DIR,"state_any_gi_50mi.csv"), index=False)

# Sensitivity by radius (Any GI)
sens = []
for r in [30,50,60,120]:
    col = f"covered_any_gi_{r}mi"
    s = zip_universe.groupby("rural_urban").apply(lambda g: pd.Series({
        "n_zips": len(g),
        "% Pop ≤R mi": pop_share(g, col, "population")
    })).reset_index()
    s["radius_mi"]=r; sens.append(s)
sens_df = pd.concat(sens, ignore_index=True)
sens_df.to_csv(os.path.join(OUT_DIR,"any_gi_sensitivity_by_radius.csv"), index=False)

# Distance distributions (Any GI)
dist = zip_universe[["dist_any_gi_mi","population","rural_urban"]].dropna()
def weighted_percentile(a, w, p):
    idx = np.argsort(a); a=a[idx]; w=w[idx]; cw=np.cumsum(w)/w.sum()
    return float(a[np.searchsorted(cw, p/100.0, side="left")])
dist_summ = []
for grp, g in dist.groupby("rural_urban"):
    dist_summ.append({
        "group": grp,
        "weighted_median_mi": weighted_percentile(g["dist_any_gi_mi"].values, g["population"].values, 50),
        "p25_mi": weighted_percentile(g["dist_any_gi_mi"].values, g["population"].values, 25),
        "p75_mi": weighted_percentile(g["dist_any_gi_mi"].values, g["population"].values, 75),
    })
pd.DataFrame(dist_summ).to_csv(os.path.join(OUT_DIR,"any_gi_distance_distribution.csv"), index=False)

print("Supporting summaries saved to ../out/")


Building supporting summaries…
Supporting summaries saved to ../out/


/var/folders/4l/64tth9h948n3dbq1py15y2qr0000gn/T/ipykernel_80122/2305359261.py:10: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  state_any50 = state.groupby(["State","rural_urban"]).apply(lambda g: pd.Series({
/var/folders/4l/64tth9h948n3dbq1py15y2qr0000gn/T/ipykernel_80122/2305359261.py:20: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  s = zip_universe.groupby("rural_urban").apply(lambda g: pd.Series({
/var/folders/